In [ ]:
import segmentation_models_pytorch as smp
from skimage.draw import disk

# --- Helper to generate Heatmaps ---
def generate_heatmap(shape, keypoints, sigma=5):
    """Generates a heatmap with Gaussian blobs at keypoint locations"""
    heatmap = np.zeros(shape, dtype=np.float32)
    for kp in keypoints:
        if kp is None: continue
        rr, cc = disk((int(kp[1]), int(kp[0])), sigma, shape=shape)
        heatmap[rr, cc] = 1.0 # Simple binary disk, can be Gaussian
        # For true Gaussian: use scipy.stats.multivariate_normal or simpler gaussian filter
    return heatmap

# --- Dataset ---
class KeypointHeatmapDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform # Use albumentations for keypoints!
        self.file_names = df['crop_file'].unique()
        
    def __len__(self):
        return len(self.file_names)
        
    def __getitem__(self, idx):
        fname = self.file_names[idx]
        group = self.df[self.df['crop_file'] == fname]
        img_path = self.img_dir / fname
        image = np.array(Image.open(img_path).convert('RGB'))
        
        # Extract points
        kps = {'Center': None, 'Top': None}
        for label in kps.keys():
            row = group[group['label'] == label]
            if not row.empty:
                kps[label] = row[['x', 'y']].values[0]

        # Resize image and adjust keypoints here if needed manually, 
        # but typically we pass to Albumentations
        
        # Create Masks (Heatmaps)
        h, w = image.shape[:2]
        mask_center = generate_heatmap((h, w), [kps['Center']])
        mask_top = generate_heatmap((h, w), [kps['Top']])
        mask = np.stack([mask_center, mask_top], axis=-1) # (H, W, 2)
        
        if self.transform:
            # Albumentations expects mask in HWC
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask'] # returns (H, W, 2)
            
        # To Tensor (C, H, W)
        image = transforms.ToTensor()(image)
        mask = torch.from_numpy(mask).permute(2, 0, 1).float()
        
        return image, mask

# --- Data Loading (Albumentations) ---
import albumentations as A
aug_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    # Add more augmentations
])

train_ds = KeypointHeatmapDataset(train_df, DATA_DIR, transform=aug_transform)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

# --- Model (U-Net) ---
model = smp.Unet(
    encoder_name="efficientnet-b0", 
    encoder_weights="imagenet", 
    in_channels=3, 
    classes=2, # Center, Top
    activation='sigmoid' # Outputs 0-1 heatmap
)
model.to(device)

# --- Training ---
criterion = nn.MSELoss() # Or nn.BCELoss() for heatmaps
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        preds = model(images)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch}, Loss: {loss.item()}")